In [1]:
%load_ext autoreload
%autoreload 2

import shutil
import sys
from pathlib import Path

from ultralytics import YOLO

# Paths come from path_config.yaml; edit that file's current_workstation to switch machines.
from path_config import PMOF_CODE_DIR, DATA_BASE_DIR, DATA_ACTIONCLS_DIR


if not PMOF_CODE_DIR.is_dir():
    raise FileNotFoundError(f"PMOF code directory not found: {PMOF_CODE_DIR}")


sys.path.insert(0, str(PMOF_CODE_DIR))

#make directories
PMOF_actioncls_base_dir = Path(DATA_ACTIONCLS_DIR)

from src.data import (
    imgid_to_annpath,
    imgid_to_imgpath,
    list_record_ids,
    read_annotation,
    recordid_to_imageids,
)


In [ ]:
record_ids = list_record_ids()
record_ids

In [ ]:
train_record_ids = ["rec4", "rec22", "rec25", "rec29", "rec30"]
val_record_ids = ["rec27"]
test_record_ids = ["rec28"]

# Restructure
Doc for Dataset Structure:  https://docs.ultralytics.com/datasets/classify#folder-structure-example

In [ ]:

for split in ["train", "val", "test"]:
    for cls in ["seated", "other"]:
        (PMOF_actioncls_base_dir / split / cls).mkdir(parents=True, exist_ok=True)

In [ ]:
def copy_images_by_action(
    record_ids: list[str],
    output_dir: Path,
    split: str,
    seated_action: str = "seated",
) -> int:
    """Copy PMOF images into an Ultralytics classification folder layout.

    Creates:
        output_dir/
        ├── {split}/
        │   ├── seated/
        │   └── other/

    An image is classified as 'seated' only if every annotated person has
    action == seated_action; if any person has another action, it is
    classified as 'other'.

    Returns the number of images processed.
    """
    output_dir = Path(output_dir)
    seated_dir = output_dir / split / "seated"
    other_dir = output_dir / split / "other"
    seated_dir.mkdir(parents=True, exist_ok=True)
    other_dir.mkdir(parents=True, exist_ok=True)

    images_counter = 0
    for record_id in record_ids:
        for image_id in recordid_to_imageids(record_id):
            images_counter += 1

            imgpath = Path(imgid_to_imgpath(image_id))
            if not imgpath.is_file():
                print(f"Warning: image not found: {imgpath}")
                continue

            annpath = imgid_to_annpath(image_id)
            person_anns = [
                ann for ann in read_annotation(annpath, image_id)
                if ann.category_name == "person"
            ]
            has_other_action = any(ann.action != seated_action for ann in person_anns)

            destination_dir = other_dir if has_other_action else seated_dir
            shutil.copy2(imgpath, destination_dir / imgpath.name)

    print(f"{split}: copied {images_counter} images")
    return images_counter


In [ ]:
for split, split_record_ids in [("val", val_record_ids), ("train", train_record_ids), ("test", test_record_ids)]:
    copy_images_by_action(split_record_ids, PMOF_actioncls_base_dir, split)


# Train Model

In [ ]:
# Load a pretrained model (recommended for training)
import torch
import torchvision.transforms as T

import cv2
import numpy as np
import torch
import albumentations as A

from albumentations.pytorch import ToTensorV2

from ultralytics import YOLO
from ultralytics.data.augment import classify_transforms
from ultralytics.data.dataset import ClassificationDataset
from ultralytics.engine.predictor import BasePredictor
from ultralytics.models.yolo.classify import ClassificationPredictor, ClassificationTrainer, ClassificationValidator

#class CustomizedDataset(ClassificationDataset):
#    """A customized dataset class for image classification with enhanced data augmentation transforms."""
#    def __init__(self, root: str, args, augment: bool = False, prefix: str = ""):
#        """Initialize a customized classification dataset with enhanced data augmentation transforms."""
#        super().__init__(root, args, augment, prefix)

#        # Add your custom training transforms here
#        train_transforms = T.Compose(
#            [
#                T.Resize((args.imgsz, args.imgsz)),
#                T.RandomRotation(degrees=args.degrees),
#                T.RandomHorizontalFlip(p=args.fliplr),
#                T.RandomVerticalFlip(p=args.flipud),

#                T.RandomErasing(p=args.erasing, inplace=True),

#                T.ColorJitter(brightness=args.hsv_v, contrast=args.hsv_v, saturation=args.hsv_s, hue=args.hsv_h),

#                T.ToTensor(),
#                T.Normalize(mean=torch.tensor(0), std=torch.tensor(1)),
#            ]
#         )

#        # Add your custom validation transforms here
#        val_transforms = T.Compose(
#            [
#                T.Resize((args.imgsz, args.imgsz)),
#                T.ToTensor(),
#                T.Normalize(mean=torch.tensor(0), std=torch.tensor(1)),
#            ]
#        )
#        self.torch_transforms = train_transforms if augment else val_transforms

class AlbumentationsClassificationTransform:
    """Albumentations wrapper for Ultralytics classification datasets."""

    def __init__(self, transform: A.Compose):
        self.transform = transform

    def __call__(self, image):
        image = np.array(image.convert("RGB"))
        augmented = self.transform(image=image)
        return augmented["image"]


class CustomizedDataset(ClassificationDataset):
    """Classification dataset with augmentations aligned to OBB training."""

    def __init__(self, root: str, args, augment: bool = False, prefix: str = ""):
        super().__init__(root, args, augment, prefix)

        train_transforms = A.Compose(
            [
                A.Resize(height=args.imgsz, width=args.imgsz),

                # A1: Geometric invariance
                A.Rotate(
                    limit=90,
                    p=1.0,
                    border_mode=cv2.BORDER_CONSTANT,
                    fill=0,
                ),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),

                # A2: Occlusion
                A.CoarseDropout(
                    num_holes_range=(1, 8),
                    hole_height_range=(0.1, 0.25),
                    hole_width_range=(0.1, 0.25),
                    fill=0,
                    p=0.5,
                ),

                # A3 & A7: grayscale/channel dropout
                A.OneOf(
                    [
                        A.ToGray(p=1.0),
                        A.ChannelDropout(p=1.0),
                    ],
                    p=0.2,
                ),

                # Color/illumination augmentations
                A.OneOf(
                    [
                        A.RandomBrightnessContrast(
                            brightness_limit=0.2,
                            contrast_limit=0.2,
                            p=0.8,
                        ),
                        A.ColorJitter(
                            brightness=0.2,
                            contrast=0.2,
                            saturation=0.2,
                            hue=0.1,
                            p=0.8,
                        ),
                        A.HueSaturationValue(
                            hue_shift_limit=20,
                            sat_shift_limit=30,
                            val_shift_limit=20,
                            p=0.8,
                        ),
                        A.RandomGamma(
                            gamma_limit=(80, 120),
                            p=0.8,
                        ),
                    ],
                    p=0.7,
                ),

                A.Normalize(mean=(0, 0, 0), std=(1, 1, 1)),
                ToTensorV2(),
            ]
        )

        val_transforms = A.Compose(
            [
                A.Resize(height=args.imgsz, width=args.imgsz),
                A.Normalize(mean=(0, 0, 0), std=(1, 1, 1)),
                ToTensorV2(),
            ]
        )

        self.torch_transforms = AlbumentationsClassificationTransform(
            train_transforms if augment else val_transforms
        )


class CustomizedTrainer(ClassificationTrainer):
    """A customized trainer class for YOLO classification models with enhanced dataset handling."""

    def build_dataset(self, img_path: str, mode: str = "train", batch=None):
        """Build a customized dataset for classification training and the validation during training."""
        return CustomizedDataset(root=img_path, args=self.args, augment=mode == "train", prefix=mode)

class CustomizedValidator(ClassificationValidator):
    """A customized validator class for YOLO classification models with enhanced dataset handling."""

    def build_dataset(self, img_path: str):
        """Build a customized dataset for classification standalone validation (no augmentation)."""
        return CustomizedDataset(root=img_path, args=self.args, augment=False, prefix=self.args.split)


class CustomizedPredictor(ClassificationPredictor):
    #def build_dataset(self, img_path: str):
    #    """Build a customized dataset for classification standalone validation (no augmentation)."""
   #     return CustomizedDataset(root=img_path, args=self.args, augment=False, prefix=self.args.split)
    def setup_source(self, source):
        BasePredictor.setup_source(self, source)
        transforms = getattr(self.model.model, "transforms", None)
        self.transforms = (
            transforms
            if isinstance(transforms, AlbumentationsClassificationTransform)
            else classify_transforms(self.imgsz)
        )


In [8]:
model = YOLO("yolo26m-cls.pt")

model.train(data=str(PMOF_actioncls_base_dir), trainer=CustomizedTrainer, epochs=1, patience=5, imgsz=640, batch=4)
model.val(data=str(PMOF_actioncls_base_dir), validator=CustomizedValidator, imgsz=1024, batch=4, split="val")
model.val(data=str(PMOF_actioncls_base_dir), validator=CustomizedValidator, imgsz=1024, batch=4, split="test")

New https://pypi.org/project/ultralytics/8.4.148 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32102MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/stella/computer_vision/PMOF_actioncls, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m-cls.pt, momentum=0.937, mosaic=1.0, mul

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78e43c532ec0>
curves: []
curves_results: []
fitness: 0.7645547986030579
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.5291095972061157, 'metrics/accuracy_top5': 1.0, 'fitness': 0.7645547986030579}
save_dir: PosixPath('/home/stella/computer_vision/enableATOsymposium/pmof-action-obbcls/runs/classify/val11')
speed: {'preprocess': 0.496962708912677, 'inference': 2.973409440089311, 'loss': 0.0011035496679877013, 'postprocess': 0.0021087380321672482}
task: 'classify'
top1: 0.5291095972061157
top5: 1.0

In [17]:
model.val(data=str(PMOF_actioncls_base_dir), validator=CustomizedValidator, imgsz=320, batch=64, split="test")

Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32102MiB)


train: /home/stella/computer_vision/PMOF_actioncls/train... found 3649 images in 2 classes ✅ 
val: /home/stella/computer_vision/PMOF_actioncls/val... found 477 images in 2 classes ✅ 
test: /home/stella/computer_vision/PMOF_actioncls/test... found 584 images in 2 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 14131.1±1355.1 MB/s, size: 2442.9 KB)
test: Scanning /home/stella/computer_vision/PMOF_actioncls/test... 584 images, 0 corrupt: 100% ━━━━━━━━━━━━ 584/584 136.1Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 10/10 1.4it/s 7.1s0.1s
                   all      0.733          1
Speed: 0.2ms preprocess, 0.3ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /home/stella/computer_vision/enableATOsymposium/pmof-action-obbcls/runs/classify/val13


ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78e40bf980a0>
curves: []
curves_results: []
fitness: 0.8664383590221405
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.732876718044281, 'metrics/accuracy_top5': 1.0, 'fitness': 0.8664383590221405}
save_dir: PosixPath('/home/stella/computer_vision/enableATOsymposium/pmof-action-obbcls/runs/classify/val13')
speed: {'preprocess': 0.20480776370028278, 'inference': 0.31994579965874137, 'loss': 8.172944894982246e-05, 'postprocess': 0.0005851489663343759}
task: 'classify'
top1: 0.732876718044281
top5: 1.0

In [21]:
#model = YOLO('/home/stella/computer_vision/enableATOsymposium/pmof-action-obbcls/runs/classify/train6/weights/best.pt')
model.predictor = None  # force Model.predict() to instantiate CustomizedPredictor instead of reusing a cached one
model.predict(source=str(PMOF_actioncls_base_dir / "val/other/rec27_001889.png"), predictor=CustomizedPredictor, visualize=True, save=True)

Saving /home/stella/computer_vision/enableATOsymposium/pmof-action-obbcls/runs/classify/predict3/rec27_001889/stage0_Conv_features.png... (32/64)
Saving /home/stella/computer_vision/enableATOsymposium/pmof-action-obbcls/runs/classify/predict3/rec27_001889/stage1_Conv_features.png... (32/128)
Saving /home/stella/computer_vision/enableATOsymposium/pmof-action-obbcls/runs/classify/predict3/rec27_001889/stage2_C3k2_features.png... (32/256)
Saving /home/stella/computer_vision/enableATOsymposium/pmof-action-obbcls/runs/classify/predict3/rec27_001889/stage3_Conv_features.png... (32/256)
Saving /home/stella/computer_vision/enableATOsymposium/pmof-action-obbcls/runs/classify/predict3/rec27_001889/stage4_C3k2_features.png... (32/512)
Saving /home/stella/computer_vision/enableATOsymposium/pmof-action-obbcls/runs/classify/predict3/rec27_001889/stage5_Conv_features.png... (32/512)
Saving /home/stella/computer_vision/enableATOsymposium/pmof-action-obbcls/runs/classify/predict3/rec27_001889/stage6_C3

[ultralytics.engine.results.Results object with attributes:
 
 boxes: None
 keypoints: None
 masks: None
 names: {0: 'other', 1: 'seated'}
 obb: None
 orig_img: array([[[  1,   1,   1],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0]],
 
        [[  1,   1,   1],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0]],
 
        [[  1,   1,   1],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0]],
 
        ...,
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 